<a href="https://colab.research.google.com/github/zeeldhorajiya1909/codsoft_tasks/blob/main/codsoft_task_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import re
import kagglehub
import os

In [ ]:
# --- 1. Data Loading ---

# Download the dataset and get the path to the downloaded directory
dataset_path_dir = kagglehub.dataset_download("hijest/genre-classification-dataset-imdb")

print("Path to dataset files directory:", dataset_path_dir)

# List contents of the downloaded directory to find the correct CSV file
print("\nContents of the downloaded dataset directory:")
for root, dirs, files in os.walk(dataset_path_dir):
    for name in files:
        print(os.path.join(root, name))
    for name in dirs:
        print(os.path.join(root, name))

# Corrected path to the training data file
# Based on the os.walk output, the files are in 'Genre Classification Dataset' subfolder
data_subfolder = 'Genre Classification Dataset'
train_file_name = 'train_data.txt'

# Construct the full path to the training data
csv_file_path = os.path.join(dataset_path_dir, data_subfolder, train_file_name)

# Load the TXT file into a Pandas DataFrame
# The dataset description on Kaggle indicates it's tab-separated
df = pd.read_csv(csv_file_path, sep=':::', engine='python', names=['id', 'title', 'genre', 'description'])

# Rename columns for clarity
# The dataset typically has 'title', 'genre', 'description'. We'll use 'description' as 'plot_summary'.
df = df.rename(columns={'description': 'plot_summary'})
df = df[['plot_summary', 'genre']]

# Handle potential missing values in plot_summary or genre
df.dropna(subset=['plot_summary', 'genre'], inplace=True)

print("\nSample Data:")
print(df.head())
print("\nUnique Genres:", df['genre'].unique())
print(f"Dataset shape after loading: {df.shape}")

Using Colab cache for faster access to the 'genre-classification-dataset-imdb' dataset.
Path to dataset files directory: /kaggle/input/genre-classification-dataset-imdb

Contents of the downloaded dataset directory:
/kaggle/input/genre-classification-dataset-imdb/Genre Classification Dataset
/kaggle/input/genre-classification-dataset-imdb/Genre Classification Dataset/description.txt
/kaggle/input/genre-classification-dataset-imdb/Genre Classification Dataset/test_data_solution.txt
/kaggle/input/genre-classification-dataset-imdb/Genre Classification Dataset/test_data.txt
/kaggle/input/genre-classification-dataset-imdb/Genre Classification Dataset/train_data.txt

Sample Data:
                                        plot_summary       genre
0   Listening in to a conversation between his do...      drama 
1   A brother and sister with a past incestuous r...   thriller 
2   As the bus empties the students for their fie...      adult 
3   To help their unemployed father make ends mee...     

In [ ]:
# --- 2. Text Preprocessing ---
def preprocess_text(text):
    # Ensure text is string, convert if not (e.g., if there were NaNs that weren't dropped)
    if not isinstance(text, str):
        return ""
    text = text.lower() # Convert to lowercase
    text = re.sub(r'[^a-z\s]', '', text) # Remove punctuation and numbers
    text = re.sub(r'\s+', ' ', text).strip() # Remove extra spaces
    return text

df['processed_plot_summary'] = df['plot_summary'].apply(preprocess_text)

print("\nProcessed Plot Summaries:")
print(df[['plot_summary', 'processed_plot_summary']].head())


Processed Plot Summaries:
                                        plot_summary  \
0   Listening in to a conversation between his do...   
1   A brother and sister with a past incestuous r...   
2   As the bus empties the students for their fie...   
3   To help their unemployed father make ends mee...   
4   The film's title refers not only to the un-re...   

                              processed_plot_summary  
0  listening in to a conversation between his doc...  
1  a brother and sister with a past incestuous re...  
2  as the bus empties the students for their fiel...  
3  to help their unemployed father make ends meet...  
4  the films title refers not only to the unrecov...  


In [ ]:
# --- 3. Feature Extraction: TF-IDF ---
# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')

# Fit and transform the processed text data
X = tfidf_vectorizer.fit_transform(df['processed_plot_summary'])
y = df['genre']

print(f"\nTF-IDF Matrix Shape: {X.shape} (Number of plots, Number of features)")



TF-IDF Matrix Shape: (54214, 5000) (Number of plots, Number of features)


In [ ]:
# --- 4. Model Training: Split Data and Train Classifier ---
# Split data into training and testing sets
# With a larger dataset, stratify=y can be re-enabled for better class distribution.
# Use a smaller test size, e.g., 0.1, due to the number of classes, or ensure enough samples per class.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nTraining samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")

# Initialize and train a Logistic Regression model
# Increased max_iter for convergence on larger dataset
logistic_regression_model = LogisticRegression(max_iter=2000, random_state=42, solver='liblinear')
logistic_regression_model.fit(X_train, y_train)

print("\nLogistic Regression model trained successfully.")



Training samples: 43371, Test samples: 10843

Logistic Regression model trained successfully.


In [ ]:
# --- 5. Model Evaluation ---
y_pred = logistic_regression_model.predict(X_test)

print("\nModel Evaluation:")
print("Accuracy:", accuracy_score(y_test, y_pred))
# Set zero_division to 'warn' or 0 or 1. Here, 0 avoids warnings for classes with no samples in test set.
print("\nClassification Report:\n", classification_report(y_test, y_pred, zero_division=0))

# --- Example of making a prediction on new data ---
new_movie_plot = "A scientist invents a time machine and travels to the past, altering the future."
processed_new_movie_plot = preprocess_text(new_movie_plot)

# Transform new text using the *fitted* TF-IDF vectorizer
new_movie_vector = tfidf_vectorizer.transform([processed_new_movie_plot])

predicted_genre = logistic_regression_model.predict(new_movie_vector)

print(f"\nNew movie plot: '{new_movie_plot}'")
print(f"Predicted genre: {predicted_genre[0]}")


Model Evaluation:
Accuracy: 0.5733653048049433

Classification Report:
                precision    recall  f1-score   support

      action        0.52      0.22      0.31       263
       adult        0.82      0.26      0.40       118
   adventure        0.70      0.12      0.21       155
   animation        0.56      0.05      0.09       100
   biography        0.00      0.00      0.00        53
      comedy        0.52      0.58      0.55      1490
       crime        0.25      0.01      0.02       101
 documentary        0.64      0.87      0.74      2619
       drama        0.53      0.77      0.63      2723
      family        0.50      0.07      0.12       157
     fantasy        0.00      0.00      0.00        65
   game-show        1.00      0.41      0.58        39
     history        0.00      0.00      0.00        49
      horror        0.65      0.55      0.59       441
       music        0.72      0.36      0.48       146
     musical        0.50      0.02      0.04  